# Class 4: Agentic AI — LLMs, Tools, and Best Practices

## What is a Large Language Model?

Large Language Models (LLMs) are the engine behind most modern AI coding assistants. Understanding how they work — and why they sometimes fail — will make you a much more effective user of these tools.

### Next-Token Prediction

At their core, LLMs do one thing: they predict the most probable next token (roughly, next word or sub-word) given all the tokens that came before. When you ask a question, the model generates a response one token at a time, each time sampling from a probability distribution over the entire vocabulary.

```
Input:  "The capital of France is"
Model:  P("Paris" | input) = 0.97
        P("Lyon"  | input) = 0.01
        P("Rome"  | input) = 0.003
        ...
Output: "Paris"
```

This simple mechanism, scaled to hundreds of billions of parameters and trained on essentially the entire written internet, produces surprisingly capable behavior.

### Training Pipeline

1. **Pre-training** — the model learns to predict the next token across a massive text corpus (books, code, Wikipedia, web pages). This is where most factual knowledge and language understanding comes from.
2. **Supervised fine-tuning** — the model is trained on curated examples of helpful conversations.
3. **RLHF / RLAIF** (Reinforcement Learning from Human/AI Feedback) — human raters (or another AI model) score responses, and the model is nudged toward responses that receive higher scores.

### Why LLMs Make Mistakes

Understanding the failure modes is just as important as knowing the capabilities:

**Hallucination**
The model generates tokens that are statistically plausible given the context, even when it has no underlying knowledge. It cannot distinguish between "I know this" and "I'm making this up" — there is no separate knowledge store to consult. Common hallucinations include invented paper titles, non-existent function names, and fabricated statistics.

**Sycophancy**
RLHF trains the model to generate responses that humans rate highly. Humans tend to rate confident, agreeable answers highly — even when they are wrong. The result is a model that is biased toward telling you what you want to hear. If you push back on a correct answer, the model may cave and adopt your (incorrect) view. Always verify independently; do not use social pressure to extract a different answer.

**Context-window limitations**
LLMs have a finite context window — the amount of text they can "see" at once. Very long conversations, large files, or complex multi-step reasoning chains can fall outside this window, causing the model to lose track of earlier information or constraints.

**No self-verification**
The model cannot run the code it writes, look up papers it cites, or otherwise check its outputs against ground truth. It generates plausible text; correctness is a side effect, not a guarantee.

:::{admonition} Key Takeaway
:class: important
LLMs are **stochastic next-token predictors**, not knowledge bases or theorem provers. They are excellent at pattern completion, style matching, and boilerplate generation. They are unreliable for precise factual recall, mathematical proof, and any task that requires verified ground truth.
:::

## Popular Agentic AI Tools

An **agentic AI** goes beyond single-turn question-answering: it can take multi-step actions — reading files, writing code, running commands — in a loop until a task is complete. The landscape is evolving rapidly; the tools below are the main players as of mid-2025.

| Tool | Maker | Interface | Best for |
|------|-------|-----------|----------|
| **Claude Code** | Anthropic | Terminal (CLI) | Agentic coding, multi-file edits, research workflows |
| **Gemini CLI** | Google | Terminal (CLI) | Similar to Claude Code, integrates with Google ecosystem |
| **GitHub Copilot** | Microsoft / GitHub | IDE extension | Inline completions, PR reviews, Copilot Workspace |
| **Cursor / Windsurf** | Anysphere / Codeium | Full IDE fork | Deep IDE integration, composer-style multi-file edits |
| **ChatGPT (GPT-4o)** | OpenAI | Web / API | Conversational tasks, Advanced Data Analysis (code interpreter) |

### Claude Code

Claude Code is a terminal-based agentic assistant built on Anthropic's Claude models. It can read and write files, run shell commands, search the codebase, and iterate toward a goal — all while keeping you in the loop for approval. Key features relevant to this course:
- **Plan mode**: generates a step-by-step plan before taking any action; you approve before execution begins.
- **Permission modes**: controls which actions the agent can take autonomously (read-only, write, shell commands).
- **Hooks**: shell commands that run automatically before/after specific agent actions.
- **MCP servers**: plug-in tools that extend the agent's capabilities (e.g., database access, web search).
- **`CLAUDE.md`**: a project-level instruction file that the agent reads at the start of every session.

### Gemini CLI

Google's terminal-based agent, structurally similar to Claude Code but backed by the Gemini model family. It integrates with Google services and has a very large context window, making it useful for tasks involving large codebases or documents.

### GitHub Copilot

The most widely used AI coding tool, embedded directly in VS Code, JetBrains, and other IDEs. It offers inline completions (single lines or whole functions), chat, and increasingly agentic "Copilot Workspace" features for multi-file tasks and PR generation.

### Cursor and Windsurf

Full IDE forks (both built on VS Code) with LLM assistance deeply embedded at every level. "Composer" mode in Cursor and "Cascade" in Windsurf allow multi-file, multi-step editing similar to terminal agents, but within a graphical IDE.

### ChatGPT / GPT-4o Advanced Data Analysis

OpenAI's conversational interface. Particularly useful for exploratory data analysis: the Advanced Data Analysis (code interpreter) feature can run Python in a sandboxed environment, generate plots, and iterate on data tasks interactively.

:::{admonition} A Word of Caution
:class: warning
Capabilities, pricing, and availability change rapidly in this space. Benchmark comparisons between models go stale within months. Choose a tool based on your actual workflow needs, not marketing claims.
:::

## Best Practices

Knowing that LLMs are stochastic next-token predictors directly implies how to use them well. The following practices apply to Claude Code specifically, but most transfer to any agentic tool.

### General Principles

**Treat the model as a capable but junior collaborator.**
It can write clean code, explain concepts, draft documentation, and suggest approaches — but it lacks domain judgment, cannot verify its own output, and will not catch its own mistakes. Your job is to review, correct, and guide.

**Keep tasks small and focused.**
Large, vague requests produce large, vague (and often incorrect) outputs. "Refactor the entire analysis pipeline" is much harder to verify than "Write a function that loads the CSV, drops NaN rows, and returns a DataFrame". Break work into reviewable units.

**Use plan mode before large changes.**
When you ask Claude Code to make substantial changes, invoke plan mode first (`/plan` or press `Shift+Tab` to toggle). Read the plan carefully before approving. This is the moment to catch misunderstandings — before files are modified.

**Version-control everything.**
Commits are cheap; undoing AI-introduced bugs is not. Commit before any significant AI-assisted change, so you can always `git checkout` to a clean state. Treat AI-generated diffs exactly like any other code review.

### Software Engineering

**Write a `CLAUDE.md` for your project.**
This file is loaded by the agent at the start of every session. Use it to describe the project structure, coding conventions, which tests to run, and any constraints (e.g., "do not modify files in `data/raw/`"). A good `CLAUDE.md` dramatically reduces misunderstandings.

**Prefer short iterative cycles.**
The feedback loop: *ask → review diff → accept or reject → commit* should be short. Do not let the agent accumulate many changes without review. A single wrong assumption early in a long chain of edits can corrupt everything that follows.

**Ask for tests alongside implementation.**
The model will happily write code that passes no tests. Explicitly ask for tests: "Write this function *and* the corresponding pytest tests." Run the tests yourself before committing.

**Manually audit security-sensitive code.**
LLMs can introduce subtle vulnerabilities: SQL injection, insecure deserialization, hard-coded credentials. Any code that handles user input, authentication, or sensitive data should be reviewed line by line before shipping.

### Research-Specific Guidance

**Reproducibility first.**
When asking the model to write analysis code, always specify: use a seeded random number generator, pin library versions, and load data from a fixed path. The model may otherwise write code that produces different results on each run.

```python
# Ask the model to write code like this, not like the version without the seed:
import numpy as np
rng = np.random.default_rng(seed=42)
```

**Verify statistical code independently.**
The model may use the wrong statistical test for your data type (e.g., a parametric test on non-normal data, or a test that assumes independence when measurements are repeated). If you are not sure the model's choice is correct, look it up.

**Do not use LLM-generated text verbatim in manuscripts.**
Use the model for scaffolding (outlines, first drafts, literature search suggestions) and editing. Do not copy-paste results sections or conclusions — the model has no access to your actual data and will generate plausible-sounding but fabricated statements.

**Verify all citations.**
LLMs hallucinate references with high confidence. Before citing any paper the model mentions, find the actual paper (via PubMed, Google Scholar, or the journal website) and confirm that it says what the model claims it says.

**Reserve AI for boilerplate; think for yourself on the science.**
Use AI to accelerate the parts of your work that are repetitive and verifiable: loading data, reshaping arrays, generating plots, writing docstrings. Reserve your critical thinking for hypothesis formulation, experimental design, and interpretation of results — the parts where being wrong matters most.

## Summary

:::{admonition} Key Points
:class: tip

1. **LLMs predict the next token** — they are not knowledge bases and cannot verify their own outputs.
2. **Hallucination and sycophancy are structural problems**, not bugs to be patched. Always verify independently.
3. **Multiple strong tools exist** (Claude Code, Gemini CLI, Copilot, Cursor); the landscape changes fast — evaluate on your actual tasks.
4. **Use plan mode and short review cycles** to catch mistakes before they compound.
5. **For research**: prioritize reproducibility, verify statistics, never copy-paste generated text into manuscripts, and always track down cited papers yourself.
:::